#Sequential Solution

In [ ]:
import math
import sys
import time

def calculate_distance(point1, point2):
    """Calculate the Euclidean distance between two points."""
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def closest_pair_divide_conquer(points):
    points.sort(key=lambda x: x[0])  # Sort points by x-coordinate
    return closest_pair_recursive(points)

def closest_pair_recursive(points):
    n = len(points)

    if n <= 3:
        return brute_force_closest_pair(points)

    mid = n // 2
    mid_point = points[mid]

    left_points = points[:mid]
    right_points = points[mid:]

    left_closest, left_distance = closest_pair_recursive(left_points)
    right_closest, right_distance = closest_pair_recursive(right_points)

    closest_pair, closest_distance = (left_closest, left_distance) if left_distance < right_distance else (right_closest, right_distance)

    strip_points = [point for point in points if abs(point[0] - mid_point[0]) < closest_distance]
    strip_points.sort(key=lambda x: x[1])

    for i in range(len(strip_points)):
        j = i + 1
        while j < len(strip_points) and strip_points[j][1] - strip_points[i][1] < closest_distance:
            dist = calculate_distance(strip_points[i], strip_points[j])
            if dist < closest_distance:
                closest_distance = dist
                closest_pair = (strip_points[i], strip_points[j])
            j += 1

    return closest_pair, closest_distance

def brute_force_closest_pair(points):
    min_distance = float('inf')
    closest_pair = None

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            distance = calculate_distance(points[i], points[j])
            if distance < min_distance:
                min_distance = distance
                closest_pair = (points[i], points[j])

    return closest_pair, min_distance

def read_points_from_txt_file(file_path):
    """Load points from an input file with comma-separated values."""
    points = []
    with open(file_path, 'r') as file:
        lines = file.readlines()

        for line in lines:
            parts = line.strip().split(',')
            if len(parts) == 2:
                x, y = map(float, parts)
                points.append((x, y))

    return points

if __name__ == '__main__':
    file_path = "points.txt"  # Specify the path to your text file

    # Read points from the text file
    points = read_points_from_txt_file(file_path)

    if not points:
        print("No data points to process.")
    else:
        # Record start time
        start_time = time.time()

        # Measure memory usage
        initial_memory = sys.getsizeof(points)

        # Find the closest pair using the divide and conquer algorithm
        closest_pair, min_distance = closest_pair_divide_conquer(points)

        # Record end time
        end_time = time.time()

        # Calculate total time
        total_time = end_time - start_time

        # Measure memory usage
        total_memory = sys.getsizeof(points) - initial_memory

        print("Closest Pair:", closest_pair)
        print("Minimum Distance:", min_distance)
        print("===============================================")
        print("Execution Time (in seconds):", total_time)
        print("Total Memory Usage (in bytes):", total_memory)


Closest Pair: ((913598.8165167435, 24414.97337182086), (913598.5221474745, 24415.526495032536))
Minimum Distance: 0.6265768539093118
Execution Time (in seconds): 12.375661134719849
Total Memory Usage (in bytes): 0


#Parallel Process

In [ ]:
import math
import sys
import time
import multiprocessing

def calculate_distance(point1, point2):
    """Calculate the Euclidean distance between two points."""
    return math.sqrt((point1[0] - point2[0])**2 + (point1[1] - point2[1])**2)

def closest_pair_divide_conquer(points):
    points.sort(key=lambda x: x[0])  # Sort points by x-coordinate
    return closest_pair_recursive(points)

def closest_pair_recursive(points):
    n = len(points)

    if n <= 3:
        return brute_force_closest_pair(points)

    mid = n // 2
    mid_point = points[mid]

    left_points = points[:mid]
    right_points = points[mid:]

    left_closest, left_distance = closest_pair_recursive(left_points)
    right_closest, right_distance = closest_pair_recursive(right_points)

    closest_pair, closest_distance = (left_closest, left_distance) if left_distance < right_distance else (right_closest, right_distance)

    strip_points = [point for point in points if abs(point[0] - mid_point[0]) < closest_distance]
    strip_points.sort(key=lambda x: x[1])

    for i in range(len(strip_points)):
        j = i + 1
        while j < len(strip_points) and strip_points[j][1] - strip_points[i][1] < closest_distance:
            dist = calculate_distance(strip_points[i], strip_points[j])
            if dist < closest_distance:
                closest_distance = dist
                closest_pair = (strip_points[i], strip_points[j])
            j += 1

    return closest_pair, closest_distance

def brute_force_closest_pair(points):
    min_distance = float('inf')
    closest_pair = None

    for i in range(len(points)):
        for j in range(i + 1, len(points)):
            distance = calculate_distance(points[i], points[j])
            if distance < min_distance:
                min_distance = distance
                closest_pair = (points[i], points[j])

    return closest_pair, min_distance

def read_points_from_txt_file(file_path):
    """Load points from an input file with comma-separated values."""
    points = []
    with open(file_path, 'r') as file:
        lines = file.readlines()

        for line in lines:
            parts = line.strip().split(',')
            if len(parts) == 2:
                x, y = map(float, parts)
                points.append((x, y))

    return points

def process_chunk(points, i, num_processes):
    chunk_size = len(points) // num_processes
    start = i * chunk_size
    end = start + chunk_size if i < num_processes - 1 else len(points)
    return closest_pair_divide_conquer(points[start:end])

if __name__ == '__main__':
    file_path = "points.txt"  # Specify the path to your text file
    num_processes = multiprocessing.cpu_count()

    # Read points from the text file
    points = read_points_from_txt_file(file_path)

    if not points:
        print("No data points to process.")
    else:
        # Record start time
        start_time = time.time()

        # Measure memory usage
        initial_memory = sys.getsizeof(points)

        # Create a pool of worker processes
        with multiprocessing.Pool(processes=num_processes) as pool:
            chunk_size = len(points) // num_processes  # Calculate the chunk size
            chunked_points = [points[i * chunk_size:(i + 1) * chunk_size] for i in range(num_processes)]
            results = pool.starmap(process_chunk, [(chunk, i, num_processes) for i, chunk in enumerate(chunked_points)])

        # Collect and merge results
        closest_pair = None
        min_distance = float('inf')
        for result in results:
            pair, distance = result
            if distance < min_distance:
                closest_pair = pair
                min_distance = distance

        # Record end time
        end_time = time.time()

        # Calculate total time
        total_time = end_time - start_time

        # Measure memory usage
        total_memory = sys.getsizeof(points) - initial_memory

        print("Closest Pair:", closest_pair)
        print("Minimum Distance:", min_distance)
        print("===============================================")
        print("Execution Time (in seconds):", total_time)
        print("Total Memory Usage (in bytes):", total_memory)



import multiprocessing

def process_chunk(chunk):
    return closest_pair_divide_conquer(chunk)

num_processes = multiprocessing.cpu_count()

with multiprocessing.Pool(processes=num_processes) as pool:
    chunk_size = len(points) // num_processes
    chunked_points = [
        points[i * chunk_size:(i + 1) * chunk_size]
        for i in range(num_processes)
    ]
    results = pool.map(process_chunk, chunked_points)


Closest Pair: ((668798.2086422974, 803798.7630007855), (668797.9113262634, 803800.6025362263))
Minimum Distance: 1.8634074868259352
Execution Time (in seconds): 4.830838441848755
Total Memory Usage (in bytes): 0
